# AVI-Adapter 的 Monte Carlo 采样梯度方向相似度分析

该 notebook 对应实验章节附录 B.6。固定 EuroSAT 中同一个训练样本和同一模型状态，执行 20 次独立 Monte Carlo 参数采样；每次重新计算训练损失并反向传播，再计算任意两次采样所得梯度向量之间的余弦相似度。

论文图仅展示实验章节规定的四个参数块：

1. R-branch projection mean；
2. R-branch scale parameter；
3. Deterministic representation parameter；
4. Text-projection scale parameter。

工程内部仍使用注册名 `AVI-Adapter` 构建随机采样基线，但论文中的方法名称统一显示为 **AVI-Adapter**。最终输出一张 2×2 等比例组合图，并同时保存为 PDF、SVG 和 PNG；PDF/SVG 中的坐标轴、刻度、标题和色条文字可直接复制。


## 0. AVI-Adapter 的训练与采样路径

工程内部通过 `AVI-AdapterMethod`/`AVI-AdapterModel` 实现 AVI-Adapter 的 Monte Carlo 训练基线。训练前向在 R 分支投影后验和文本投影后验中进行随机采样，并将采样噪声经损失反向传播到变分参数和确定性表示参数。

本 notebook 将每次前向的 Monte Carlo 样本数固定为 1，并在同一输入与同一模型状态下独立重复 20 次，从而只考察随机采样造成的单步梯度方向变化。


## 1. 基本配置

需要按你的本地路径修改：

- `PROJECT_ROOT`：项目中的 `MMRL/` 目录
- `DATA_ROOT`：数据集根目录
- `DATASET_NAME`：用于取固定图像的数据集，例如 `eurosat`
- `MODEL_DIR`：可选；填入训练好的 checkpoint 目录时，会先加载模型再分析梯度

In [1]:
from pathlib import Path
import os

# ===== 必改：指向你的 MMRL 目录 =====
PROJECT_ROOT = Path("/root/autodl-tmp/MMRL").expanduser().resolve()

# ===== 数据与实验配置 =====
DATA_ROOT = Path("DATASETS").expanduser()   # 也可以写绝对路径
DATASET_NAME = "eurosat"
PROTOCOL = "FS"
SHOTS = 16
SEED = 1
BACKBONE = "ViT-B/16"

METHOD_CONFIG = "configs/methods/bayesrt_mmrl.yaml"
PROTOCOL_CONFIG = "configs/protocols/fs.yaml"
RUNTIME_CONFIG = "configs/runtime/mmrl_family.yaml"

# 可选：加载已经训练好的 AVI-Adapter（工程注册名 BayesRTMMRL）checkpoint。
# 目录通常类似：
# output_refactor/BayesRTMMRL/FS/fewshot_train/eurosat/shots_8/ViT-B-16/default/seed1
MODEL_DIR = None
LOAD_EPOCH = None

# 梯度分析配置
NUM_GRAD_SAMPLES = 20
FORCE_N_MC_PER_FORWARD = 1       # 推荐为 1：每次 backward 对应一个 posterior sample
USE_POSTERIOR_MEAN_CONTROL = False
LOSS_KEY = "total"               # 可选: "total" 或 "data_term"；data_term 更能隔离采样噪声
FIX_SINGLE_IMAGE = True
BATCH_INDEX = 0

# 输出目录
ANALYSIS_DIR = PROJECT_ROOT / "output_refactor" / "analysis"  / "gradient_analysis" / "avi_adapter_paper_ready"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ANALYSIS_DIR:", ANALYSIS_DIR)

# 论文显示名称与图形导出设置。
METHOD_DISPLAY_NAME = "AVI-Adapter"
PAPER_FIGURE_FORMATS = ("pdf", "svg", "png")
PAPER_PARAMETER_ORDER = ["R_mean", "R_rho", "representation_learner", "T_rho"]
PAPER_PARAMETER_LABELS = {
    "R_mean": "(a) R-branch projection mean",
    "R_rho": "(b) R-branch scale parameter",
    "representation_learner": "(c) Deterministic representation parameter",
    "T_rho": "(d) Text-projection scale parameter",
}


PROJECT_ROOT: /root/autodl-tmp/MMRL
ANALYSIS_DIR: /root/autodl-tmp/MMRL/output_refactor/analysis/gradient_analysis/avi_adapter_paper_ready


## 2. 导入项目并构建 trainer

这里复用项目的 `run.py` 入口逻辑和 `RefactorRunner`，但不会调用 `trainer.train()`，只构建模型、数据加载器和优化器。

In [2]:
import sys
from types import SimpleNamespace

assert PROJECT_ROOT.exists(), f"PROJECT_ROOT 不存在: {PROJECT_ROOT}"
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "axes.unicode_minus": False,
})

from dassl.engine import build_trainer
from dassl.utils import set_random_seed
from core.config import setup_cfg
from run import _import_runtime_modules

_import_runtime_modules()

set_random_seed(SEED)

args = SimpleNamespace(
    root=str(DATA_ROOT),
    output_dir=str(ANALYSIS_DIR / "tmp_build"),
    dataset_config_file=f"configs/datasets/{DATASET_NAME}.yaml",
    method_config_file=METHOD_CONFIG,
    protocol_config_file=PROTOCOL_CONFIG,
    runtime_config_file=RUNTIME_CONFIG,
    exp_config="",
    method="BayesRTMMRL",
    protocol=PROTOCOL,
    exec_mode="online",
    seed=SEED,
    trainer="RefactorRunner",
    eval_only=False,
    model_dir="" if MODEL_DIR is None else str(MODEL_DIR),
    load_epoch=LOAD_EPOCH,
    no_train=True,
    opts=[
        "DATASET.NUM_SHOTS", str(SHOTS),
        "DATASET.SUBSAMPLE_CLASSES", "all",
        "MODEL.BACKBONE.NAME", BACKBONE,
        "HPO.ENABLED", "False",
    ],
)

cfg = setup_cfg(args)
cfg.defrost()
cfg.HPO.ENABLED = False
cfg.freeze()

trainer = build_trainer(cfg)

if MODEL_DIR is not None:
    print(f"Loading checkpoint from: {MODEL_DIR}")
    trainer.load_model(str(MODEL_DIR), epoch=LOAD_EPOCH)

method = trainer.method
model = method.model
device = trainer.device

method.n_mc_train = int(FORCE_N_MC_PER_FORWARD)

print("device:", device)
print("method.n_mc_train forced to:", method.n_mc_train)
print("BAYES_R_ENABLED:", bool(cfg.BAYESRT_MMRL.BAYES_R_ENABLED))
print("BAYES_T_ENABLED:", bool(cfg.BAYESRT_MMRL.BAYES_T_ENABLED))
print("LOSS_KEY:", LOSS_KEY)

[WARN] optional import failed: datasets.deepdird: No module named 'datasets.deepdird'
Loading trainer: RefactorRunner
Loading dataset: EuroSAT
Reading split from /root/autodl-tmp/MMRL/DATASETS/eurosat/split_zhou_EuroSAT.json
Loading preprocessed few-shot data from /root/autodl-tmp/MMRL/DATASETS/eurosat/split_fewshot/shot_16-seed_1.pkl
Building transform_train
+ random resized crop (size=(224, 224), scale=(0.5, 1))
+ random flip
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
Building transform_test
+ resize the smaller edge to 224
+ 224x224 center crop
+ to torch tensor of range [0, 1]
+ normalization (mean=[0.48145466, 0.4578275, 0.40821073], std=[0.26862954, 0.26130258, 0.27577711])
---------  -------
Dataset    EuroSAT
# classes  10
# train_x  160
# val      40
# test     8,100
---------  -------
[BayesRTMMRL] trainable params: {'representation_learner.compound_rep_tokens_r2vproj.1.bias', 'represe

/root/autodl-tmp/MMRL/trainers/refactor_runner.py:75: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler() if prec == "amp" else None


## 3. 取固定图像 batch

默认取训练 loader 的第一个 batch，并裁剪成单张图像，以匹配“固定一张图像，重复 20 次采样”的分析设定。

In [3]:
def _first_existing_attr(obj, names):
    for name in names:
        if hasattr(obj, name):
            value = getattr(obj, name)
            if value is not None:
                return value
    return None

def get_train_loader(trainer):
    candidates = [
        "train_loader_x",
        "train_loader",
        "train_loader_u",
    ]
    loader = _first_existing_attr(trainer, candidates)
    if loader is not None:
        return loader

    dm = getattr(trainer, "dm", None)
    if dm is not None:
        loader = _first_existing_attr(dm, candidates)
        if loader is not None:
            return loader

    raise RuntimeError(
        "找不到训练 dataloader。请检查 trainer.train_loader_x 或 trainer.dm.train_loader_x。"
    )

def slice_batch_to_one(batch):
    out = {}
    for k, v in batch.items():
        if torch.is_tensor(v) and v.shape[0] > 0:
            out[k] = v[:1]
        else:
            out[k] = v
    return out

loader = get_train_loader(trainer)

batch = None
for idx, candidate in enumerate(loader):
    if idx == BATCH_INDEX:
        batch = candidate
        break

assert batch is not None, f"无法取得 batch index={BATCH_INDEX}"

if FIX_SINGLE_IMAGE:
    batch = slice_batch_to_one(batch)

payload = {
    "img": batch["img"].to(device),
    "label": batch["label"].to(device),
}

print("img shape:", tuple(payload["img"].shape))
print("label:", payload["label"].detach().cpu().tolist())

img shape: (1, 3, 224, 224)
label: [8]


## 4. 定义梯度参数块

工程参数名与论文图名称的对应关系如下：

- `R_mean`：R-branch projection mean；
- `R_rho`：R-branch scale parameter；
- `representation_learner`：Deterministic representation parameter；
- `T_rho`：Text-projection scale parameter。

`T_mean` 可能存在于某些配置中，但不属于实验章节图 4 的四个展示项，因此不会进入主论文组合图。


In [4]:
def named_trainable_parameters(module):
    return [(n, p) for n, p in module.named_parameters() if p.requires_grad]

def collect_param_groups(method):
    model = method.model
    groups = {}

    bayes_r = getattr(getattr(model, "image_encoder", None), "bayes_proj_rep", None)
    if bayes_r is not None:
        if isinstance(getattr(bayes_r, "posterior_mean", None), torch.nn.Parameter):
            groups["R_mean"] = [bayes_r.posterior_mean]
        if isinstance(getattr(bayes_r, "posterior_rho", None), torch.nn.Parameter):
            groups["R_rho"] = [bayes_r.posterior_rho]

    text_post = getattr(model, "text_posterior", None)
    if text_post is not None:
        if isinstance(getattr(text_post, "posterior_mean", None), torch.nn.Parameter):
            groups["T_mean"] = [text_post.posterior_mean]
        if isinstance(getattr(text_post, "posterior_rho", None), torch.nn.Parameter):
            groups["T_rho"] = [text_post.posterior_rho]

    rep_learner = getattr(model, "representation_learner", None)
    if rep_learner is not None:
        params = [p for _, p in named_trainable_parameters(rep_learner)]
        if params:
            groups["representation_learner"] = params

    # 兜底：如果 R 侧 Bayesian 被关掉，项目中可能训练 deterministic proj_rep
    det_visual = getattr(getattr(model, "image_encoder", None), "visual", None)
    if det_visual is not None and hasattr(det_visual, "proj_rep"):
        p = det_visual.proj_rep
        if isinstance(p, torch.nn.Parameter) and p.requires_grad:
            groups["visual_proj_rep"] = [p]

    return groups

def flatten_grad(params):
    chunks = []
    total_numel = 0
    missing = 0
    for p in params:
        total_numel += p.numel()
        if p.grad is None:
            missing += p.numel()
            continue
        chunks.append(p.grad.detach().float().reshape(-1).cpu())
    if not chunks:
        return None, {"numel": total_numel, "missing_grad_numel": missing}
    return torch.cat(chunks, dim=0), {"numel": total_numel, "missing_grad_numel": missing}

def zero_all_grads(trainer, method):
    if hasattr(trainer, "optim") and trainer.optim is not None:
        trainer.optim.zero_grad(set_to_none=True)
    else:
        method.model.zero_grad(set_to_none=True)

def forward_train_like_project(method, payload, use_posterior_mean=False, num_samples=1):
    # 等价复刻 BayesRTMMRLMethod.forward_train，但允许控制 use_posterior_mean。
    image = payload["img"].to(method.device)
    label = payload["label"].to(method.device)

    with torch.no_grad():
        img_ref = method.image_encoder_clip(image.type(method.dtype))
        img_ref = torch.nn.functional.normalize(img_ref, dim=-1)

    out = method.model.forward_joint(
        image=image,
        num_samples=int(num_samples),
        use_posterior_mean=bool(use_posterior_mean),
    )
    return method._build_train_outputs(label, img_ref, out)

param_groups = collect_param_groups(method)
print("Parameter groups:")
for name, params in param_groups.items():
    n_params = sum(p.numel() for p in params)
    print(f"  {name:24s} tensors={len(params):3d} numel={n_params:,}")

Parameter groups:
  R_mean                   tensors=  1 numel=393,216
  R_rho                    tensors=  1 numel=512
  T_rho                    tensors=  1 numel=512
  representation_learner   tensors= 29 numel=4,599,040


## 5. 重复采样并提取梯度

默认每次 backward 前都会设置不同随机种子，保证每次 posterior sample 不同，同时输入图像不变。

In [5]:
def collect_gradients(
    trainer,
    method,
    payload,
    param_groups,
    num_repeats=20,
    loss_key="total",
    num_mc_per_forward=1,
    use_posterior_mean_control=False,
    base_seed=12345,
):
    method.model.train()

    records = []
    grad_bank = {name: [] for name in param_groups}
    meta_bank = {name: [] for name in param_groups}

    for i in range(int(num_repeats)):
        seed_i = int(base_seed + i)
        torch.manual_seed(seed_i)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed_i)

        zero_all_grads(trainer, method)

        outputs = forward_train_like_project(
            method,
            payload,
            use_posterior_mean=use_posterior_mean_control,
            num_samples=num_mc_per_forward,
        )

        if loss_key not in outputs.losses:
            raise KeyError(f"loss_key={loss_key!r} 不在 outputs.losses 中。可选: {list(outputs.losses.keys())}")

        loss = outputs.losses[loss_key]
        loss.backward()

        row = {"repeat": i, "seed": seed_i, "loss_key": loss_key, "loss": float(loss.detach().cpu())}
        for k, v in outputs.losses.items():
            if torch.is_tensor(v) and v.numel() == 1:
                row[k] = float(v.detach().cpu())
        records.append(row)

        for group_name, params in param_groups.items():
            g, meta = flatten_grad(params)
            if g is not None:
                grad_bank[group_name].append(g)
            meta_bank[group_name].append(meta)

        del outputs, loss
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return grad_bank, meta_bank, pd.DataFrame(records)

grad_bank, meta_bank, loss_df = collect_gradients(
    trainer=trainer,
    method=method,
    payload=payload,
    param_groups=param_groups,
    num_repeats=NUM_GRAD_SAMPLES,
    loss_key=LOSS_KEY,
    num_mc_per_forward=FORCE_N_MC_PER_FORWARD,
    use_posterior_mean_control=USE_POSTERIOR_MEAN_CONTROL,
    base_seed=SEED * 100000 + 17,
)

display(loss_df.head())
print(loss_df[["loss"]].describe())

,repeat,seed,loss_key,loss,loss_main,loss_rep,loss_cos_img,loss_cos_text,data_term,raw_kl_r,raw_kl_t,kl_r_term,kl_t_term,kl_beta,kl_normalizer,total
0,0,100017,total,2.551740,2.731012,1.620707,0.062591,0.245047,2.551740,1.396984e-09,3.725290e-09,1.396984e-13,7.105427e-16,1.0,1.0,2.551740
1,1,100018,total,2.455522,2.703026,1.365283,0.062591,0.245047,2.455522,1.396984e-09,3.725290e-09,1.396984e-13,7.105427e-16,1.0,1.0,2.455522
2,2,100019,total,2.113464,2.689800,0.255950,0.062591,0.245047,2.113464,1.396984e-09,3.725290e-09,1.396984e-13,7.105427e-16,1.0,1.0,2.113464
3,3,100020,total,3.450638,2.751693,4.568779,0.062591,0.245047,3.450638,1.396984e-09,3.725290e-09,1.396984e-13,7.105427e-16,1.0,1.0,3.450638
4,4,100021,total,2.484106,2.620393,1.653375,0.062591,0.245047,2.484106,1.396984e-09,3.725290e-09,1.396984e-13,7.105427e-16,1.0,1.0,2.484106


            loss
count  20.000000
mean    2.964707
std     0.406036
min     2.113464
25%     2.625228
50%     3.094280
75%     3.271123
max     3.483661


## 6. 计算两两余弦相似度矩阵与摘要指标

In [6]:
def cosine_similarity_matrix(grad_list, eps=1e-12):
    if len(grad_list) == 0:
        return None
    G = torch.stack(grad_list, dim=0).float()
    norms = G.norm(dim=1, keepdim=True).clamp_min(eps)
    G = G / norms
    return (G @ G.T).cpu().numpy()

def offdiag_values(mat):
    n = mat.shape[0]
    mask = ~np.eye(n, dtype=bool)
    return mat[mask]

def summarize_cosine_matrix(name, mat):
    vals = offdiag_values(mat)
    return {
        "group": name,
        "n": mat.shape[0],
        "offdiag_mean": float(np.mean(vals)),
        "offdiag_std": float(np.std(vals)),
        "offdiag_min": float(np.min(vals)),
        "offdiag_q25": float(np.quantile(vals, 0.25)),
        "offdiag_median": float(np.median(vals)),
        "offdiag_q75": float(np.quantile(vals, 0.75)),
        "offdiag_max": float(np.max(vals)),
        "frac_negative": float(np.mean(vals < 0.0)),
    }

cos_mats = {}
summary_rows = []

for group_name, grads in grad_bank.items():
    mat = cosine_similarity_matrix(grads)
    if mat is None:
        continue
    cos_mats[group_name] = mat
    summary_rows.append(summarize_cosine_matrix(group_name, mat))

summary_df = pd.DataFrame(summary_rows).sort_values("offdiag_mean")
display(summary_df)

loss_csv = ANALYSIS_DIR / f"loss_records_{DATASET_NAME}_shots{SHOTS}_seed{SEED}.csv"
summary_csv = ANALYSIS_DIR / f"gradient_cosine_summary_{DATASET_NAME}_shots{SHOTS}_seed{SEED}.csv"
loss_df.to_csv(loss_csv, index=False)
summary_df.to_csv(summary_csv, index=False)

print("Saved:", loss_csv)
print("Saved:", summary_csv)

,group,n,offdiag_mean,offdiag_std,offdiag_min,offdiag_q25,offdiag_median,offdiag_q75,offdiag_max,frac_negative
2,T_rho,20,-0.014454,0.309609,-0.641899,-0.264304,-0.039191,0.244387,0.661754,0.547368
1,R_rho,20,0.000822,0.072446,-0.172448,-0.051811,0.006408,0.046972,0.206715,0.457895
3,representation_learner,20,0.731068,0.066029,0.503846,0.687093,0.730394,0.770281,0.920057,0.000000
0,R_mean,20,0.885388,0.062044,0.677472,0.858388,0.890778,0.932794,0.986573,0.000000


Saved: /root/autodl-tmp/MMRL/output_refactor/analysis/gradient_analysis/avi_adapter_paper_ready/loss_records_eurosat_shots16_seed1.csv
Saved: /root/autodl-tmp/MMRL/output_refactor/analysis/gradient_analysis/avi_adapter_paper_ready/gradient_cosine_summary_eurosat_shots16_seed1.csv


## 7. 分别绘制主论文图 4 的四幅梯度方向相似度矩阵

四幅矩阵不再排成 2×2，而是按照实验章节 B.6 的顺序分别保存为独立的 PDF、SVG 和 PNG：

1. R 分支投影均值；
2. R 分支尺度参数；
3. 确定性表示参数；
4. 文本投影尺度参数。

四幅图统一使用色条范围 `[-1, 1]`。横纵坐标均表示 20 次 Monte Carlo 采样编号；对角线恒为 1，非对角线越接近 1 表示不同采样下的梯度方向越一致。图像采用 `pcolormesh` 绘制，因此 PDF/SVG 中的矩阵、坐标轴和文字均保持矢量形式。


In [7]:
def save_figure_bundle(fig, output_path, dpi=300):
    """Save PNG preview and vector PDF/SVG with identical layout."""
    output_path = Path(output_path)
    stem = output_path.with_suffix("")
    saved = {}
    for fmt in PAPER_FIGURE_FORMATS:
        target = stem.with_suffix(f".{fmt}")
        kwargs = {"bbox_inches": "tight"}
        if fmt == "png":
            kwargs["dpi"] = int(dpi)
        fig.savefig(target, **kwargs)
        saved[fmt] = str(target)
    return saved


def _set_mc_ticks(ax, n):
    # 论文中以 1--20 编号，避免显示 Python 的 0--19 索引。
    centers = np.arange(n) + 0.5
    labels = np.arange(1, n + 1)
    if n <= 20:
        ax.set_xticks(centers)
        ax.set_yticks(centers)
        ax.set_xticklabels(labels, fontsize=7)
        ax.set_yticklabels(labels, fontsize=7)
    else:
        step = max(1, int(np.ceil(n / 20)))
        idx = np.arange(0, n, step)
        ax.set_xticks(idx + 0.5)
        ax.set_yticks(idx + 0.5)
        ax.set_xticklabels(labels[idx], fontsize=7)
        ax.set_yticklabels(labels[idx], fontsize=7)


PAPER_PARAMETER_FILE_TAGS = {
    "R_mean": "AVI-Adapter_R_branch_projection_mean",
    "R_rho": "AVI-Adapter_R_branch_scale_parameter",
    "representation_learner": "AVI-Adapter_deterministic_representation_parameter",
    "T_rho": "AVI-Adapter_text_projection_scale_parameter",
}


def plot_individual_gradient_similarity_figures(cos_mats):
    """Save each B.6 gradient-similarity matrix as an independent figure."""
    figure_rows = []

    for group_name in PAPER_PARAMETER_ORDER:
        mat = cos_mats.get(group_name)
        if mat is None:
            print(f"[WARN] Missing gradient block: {group_name}")
            continue

        mat = np.asarray(mat, dtype=float)
        n = mat.shape[0]
        edges = np.arange(n + 1)

        fig, ax = plt.subplots(figsize=(5.6, 5.0))
        ax.set_box_aspect(1)

        mesh = ax.pcolormesh(
            edges,
            edges,
            mat,
            vmin=-1.0,
            vmax=1.0,
            shading="flat",
        )
        ax.invert_yaxis()
        _set_mc_ticks(ax, n)
        ax.set_xlabel("Monte Carlo sample")
        ax.set_ylabel("Monte Carlo sample")
        ax.set_title(
            f"{PAPER_PARAMETER_LABELS[group_name]}\n"
            f"{METHOD_DISPLAY_NAME}, EuroSAT, fixed training sample",
            fontsize=10.5,
        )

        cbar = fig.colorbar(mesh, ax=ax, pad=0.03)
        cbar.set_label("Gradient cosine similarity")
        fig.tight_layout()

        file_tag = PAPER_PARAMETER_FILE_TAGS[group_name]
        panel_path = ANALYSIS_DIR / (
            f"Figure4_{file_tag}_{DATASET_NAME}_"
            f"{NUM_GRAD_SAMPLES}MC_seed{SEED}.png"
        )
        files = save_figure_bundle(fig, panel_path)
        plt.close(fig)

        figure_rows.append({
            "parameter_block": group_name,
            "label": PAPER_PARAMETER_LABELS[group_name],
            "method": METHOD_DISPLAY_NAME,
            "dataset": DATASET_NAME,
            "num_mc_samples": int(NUM_GRAD_SAMPLES),
            "seed": int(SEED),
            **files,
        })

    return figure_rows


individual_figure_rows = plot_individual_gradient_similarity_figures(cos_mats)
individual_figure_df = pd.DataFrame(individual_figure_rows)

individual_figure_csv = ANALYSIS_DIR / "Figure4_separate_figure_files.csv"
individual_figure_df.to_csv(individual_figure_csv, index=False)

print("Saved:", individual_figure_csv)
display(individual_figure_df)


Saved: /root/autodl-tmp/MMRL/output_refactor/analysis/gradient_analysis/avi_adapter_paper_ready/Figure4_separate_figure_files.csv


,parameter_block,label,method,dataset,num_mc_samples,seed,pdf,svg,png
0,R_mean,(a) R-branch projection mean,AVI-Adapter,eurosat,20,1,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...
1,R_rho,(b) R-branch scale parameter,AVI-Adapter,eurosat,20,1,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...
2,representation_learner,(c) Deterministic representation parameter,AVI-Adapter,eurosat,20,1,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...
3,T_rho,(d) Text-projection scale parameter,AVI-Adapter,eurosat,20,1,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...,/root/autodl-tmp/MMRL/output_refactor/analysis...


## 8. 可选：把所有矩阵保存为 `.npy`

保存后可在其他脚本或论文绘图脚本中重新加载。

In [8]:
for group_name, mat in cos_mats.items():
    npy_path = ANALYSIS_DIR / f"cosmat_{group_name}_{DATASET_NAME}_shots{SHOTS}_seed{SEED}.npy"
    np.save(npy_path, mat)
    print("Saved:", npy_path)

Saved: /root/autodl-tmp/MMRL/output_refactor/analysis/gradient_analysis/avi_adapter_paper_ready/cosmat_R_mean_eurosat_shots16_seed1.npy
Saved: /root/autodl-tmp/MMRL/output_refactor/analysis/gradient_analysis/avi_adapter_paper_ready/cosmat_R_rho_eurosat_shots16_seed1.npy
Saved: /root/autodl-tmp/MMRL/output_refactor/analysis/gradient_analysis/avi_adapter_paper_ready/cosmat_T_rho_eurosat_shots16_seed1.npy
Saved: /root/autodl-tmp/MMRL/output_refactor/analysis/gradient_analysis/avi_adapter_paper_ready/cosmat_representation_learner_eurosat_shots16_seed1.npy


## 9. 与实验章节一致的结果解释

图 4 展示同一输入样本在 20 次 Monte Carlo 采样下的梯度方向相似度矩阵。R 分支尺度参数或文本投影尺度参数出现较低甚至为负的非对角相似度，表示随机采样会使后验尺度参数的单步更新方向发生明显波动；确定性表示参数的相似度则反映采样噪声是否进一步传播至表示学习模块。

因此，图中方法名称统一为 AVI-Adapter，用于表示采用 Monte Carlo 采样近似 ELBO 的双分支变分基线；DAVI-Adapter 不在训练阶段进行随机采样，其稳定性优势由该对照分析间接说明。
